# Model Evaluation
---

### Imports

In [ ]:
from pathlib import Path
from src.utils import get_device

import matplotlib.pyplot as plt
import pandas as pd
import torch

from src.dataset import read_xray
from src.model import MyModel

### Paths and device

In [ ]:
PROJECT_DIR = Path.cwd()
TEST_CSV = PROJECT_DIR / "data" / "csv" / "test.csv"
CHECKPOINT = PROJECT_DIR / "sessions" / "best_model.pth"
DEVICE = get_device()

### Load test data

In [ ]:
if not TEST_CSV.is_file():
    raise FileNotFoundError(f"Test CSV not found: {TEST_CSV}")
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Model checkpoint not found: {CHECKPOINT}")

test_df = pd.read_csv(TEST_CSV, keep_default_na=False)
if test_df.empty:
    raise ValueError(f"The test CSV contains no images: {TEST_CSV}")

print(f"{len(test_df)} test images | Device: {DEVICE}")

### Load model and define prediction helper

In [ ]:
model = MyModel().to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.eval()


def predict_mask(image, threshold=0.5):
    """Return the binary segmentation mask for a preprocessed X-ray."""
    x = torch.from_numpy(image).unsqueeze(0).to(DEVICE)  # shape: (1, 1, H, W)

    with torch.inference_mode():
        probs = torch.sigmoid(model(x))[0, 0].cpu().numpy()
    return probs >= threshold


def display_prediction(index, threshold=0.5):
    """Show one test X-ray, its predicted mask and an overlay."""
    path = Path(test_df.iloc[index]["xrays"])
    image = read_xray(path)
    mask = predict_mask(image, threshold)
    display_image = image[0]

    plt.figure(figsize=(16, 8))

    plt.subplot(1, 3, 1)
    plt.imshow(display_image, cmap="gray")
    plt.title("X-ray")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(mask, cmap="gray")
    plt.title(f"Predicted mask (threshold {threshold:.2f})")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(display_image, cmap="gray")
    plt.imshow(mask, cmap="autumn", alpha=0.45)
    plt.title("Mask overlay")
    plt.axis("off")

    plt.suptitle(f"Test image {index}: {path.name}")
    plt.tight_layout()
    plt.show()

### Save all mask overlays

Save an overlay image for every test X-ray to `sessions/mask_overlays/`.

In [ ]:
OVERLAY_DIR = PROJECT_DIR / "sessions" / "mask_overlays"
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

for path in test_df["xrays"]:
    image_path = Path(path)
    image = read_xray(path)
    mask = predict_mask(image)

    plt.imshow(image[0], cmap="gray")
    plt.imshow(mask, cmap="autumn", alpha=0.45)
    plt.axis("off")
    plt.savefig(OVERLAY_DIR / f"{image_path.stem}.png", bbox_inches="tight", pad_inches=0)
    plt.close()

print(f"Overlays saved to {OVERLAY_DIR}")

### Inspect predictions

In [ ]:
image_index = 0

In [ ]:
# Change the index and rerun this cell to inspect the next image.
display_prediction(image_index)
image_index += 1